In [1]:
from pathlib import Path
import sys


sys.path.append(str(Path().resolve().parent))

from utilities.src import build_df



  



In [2]:
BASE_DIR = Path().resolve()
PROJECT_ROOT = BASE_DIR.parent   #to go un up the the root
DATA_DIR = PROJECT_ROOT / "data"

# Root directory of dataset
root = Path(DATA_DIR)


In [3]:
templates_df=build_df(root)

print(templates_df.head())
print("number of samples:", templates_df["sample_id"].nunique())
print(templates_df.groupby("sample_id").size().head())

   time index     acc_x     acc_y     acc_z     gyr_x     gyr_y     gyr_z  \
0         314 -9.675384 -1.665096  0.473116 -0.016972  0.022434  0.003591   
1         315 -9.660426 -1.687629  0.488249 -0.007946 -0.000952 -0.002020   
2         316 -9.660451 -1.702569  0.488208 -0.004190 -0.001160 -0.012884   
3         317 -9.660356 -1.657827  0.510743 -0.004123  0.006832 -0.016450   
4         318 -9.645445 -1.687700  0.488502 -0.004284  0.007983 -0.003784   

      mag_x     mag_y     mag_z subject exercise unit execution_type  \
0  0.588639  0.457257 -0.081429      s1       e1   u1        correct   
1  0.587563  0.457905 -0.081475      s1       e1   u1        correct   
2  0.588892  0.457524 -0.081558      s1       e1   u1        correct   
3  0.587898  0.457243 -0.080656      s1       e1   u1        correct   
4  0.588329  0.457522 -0.081974      s1       e1   u1        correct   

   sample_id  
0          0  
1          0  
2          0  
3          0  
4          0  
number of samp

In [ ]:
feature_cols = [
    "acc_x", "acc_y", "acc_z",
    "gyr_x", "gyr_y", "gyr_z",
    "mag_x", "mag_y", "mag_z"
]

In [5]:
sample = templates_df[templates_df["sample_id"] == 1]

X = sample[feature_cols].values
y = sample["execution_type"].iloc[0]

print(X.shape)
print(y)

(106, 9)
fast


In [28]:
import pandas as pd
 
 
def analyze_segment_lengths(templates_df):
    """
    For each sample_id, count the number of timesteps.
    Then summarize by execution_type: mean, std, min, max.
    """
 
    # Step 1: count timesteps per sample
    lengths = (
        templates_df
        .groupby(["sample_id", "execution_type", "subject", "exercise", "unit"])
        .size()
        .reset_index(name="n_timesteps")
    )
 
    # Step 2: summary stats grouped by execution type
    summary = (
        lengths
        .groupby("execution_type")["n_timesteps"]
        .agg(["mean", "std", "min", "max", "count"])
        .round(1)
    )
 
    print("=== Timesteps per execution type ===")
    print(summary)
    print()
 
    # Step 3: per exercise breakdown (to see if pattern holds across all 8)
    per_exercise_mean = (
        lengths
        .groupby(["exercise", "execution_type"])["n_timesteps"]
        .mean()
        .round(1)
        .unstack("execution_type")
    )
    per_exercise_mean.columns = [f"{c}_mean" for c in per_exercise_mean.columns]
 
    per_exercise_max = (
        lengths
        .groupby(["exercise", "execution_type"])["n_timesteps"]
        .max()
        .unstack("execution_type")
    )
    per_exercise_max.columns = [f"{c}_max" for c in per_exercise_max.columns]
 
    per_exercise_min = (
        lengths
        .groupby(["exercise", "execution_type"])["n_timesteps"]
        .min()
        .unstack("execution_type")
    )
    per_exercise_min.columns = [f"{c}_min" for c in per_exercise_min.columns]
 
    per_exercise = pd.concat([per_exercise_mean, per_exercise_max, per_exercise_min], axis=1).sort_index(axis=1)
 
    print("=== Mean, min and max timesteps per exercise × execution type ===")
    print(per_exercise)
 
    return lengths, summary, per_exercise

In [29]:
lengths, summary, per_exercise = analyze_segment_lengths(templates_df)

=== Timesteps per execution type ===
                 mean   std  min  max  count
execution_type                              
correct         142.3  30.6   98  218    200
fast             71.8  17.2   49  116    200
low_amplitude   125.2  27.2   84  203    200

=== Mean, min and max timesteps per exercise × execution type ===
          correct_max  correct_mean  correct_min  fast_max  fast_mean  \
exercise                                                                
e1                215         170.4          146       113       84.2   
e2                200         158.0          137        91       79.6   
e3                168         143.6          121       113       80.6   
e4                159         128.2           98        65       58.8   
e5                148         127.2           98        75       62.0   
e6                218         142.6          114       116       78.8   
e7                176         136.0           99        73       65.6   
e8            

In [27]:
sample_x = templates_df[templates_df["sample_id"] == 301]
print(len(sample_x))                          
print(sample_x["execution_type"].iloc[0])    
print(sample_x["time index"].min(), "→", sample_x["time index"].max())

52
fast
531 → 582
